# Chapter 17 &mdash; Carroll's Wise Young Pigs, and Why Every Premise Earns Its Place

**Concept 9 of the Chapter 17 decomposition:** *Carroll's Wise Young Pigs, and Why Every Premise Earns Its Place*

Nine premises over thirteen variables; the refutation is still one node, and removing any single premise breaks the proof.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter17-BDD/Concept-Carroll-Wise-Young-Pigs/Concept-Carroll-Wise-Young-Pigs.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Carroll's longer puzzle, and the one that shows why the method is worth having.

Nine premises, thirteen variables, and this conclusion:

> **Wise young pigs do not go up in balloons.**

The premises are the usual Carroll machinery &mdash; pigs that are giddy are
respected, wise creatures with balloons carry umbrellas, the fat and ridiculous who do
not dance on tightropes lunch in public. Unlike the crocodiles, **no reader can see
the answer by staring at it.** There are $2^{13} = 8192$ assignments and 1251 of them
satisfy the premises; the conclusion is not going to be read off a table.

The refutation is one node, exactly as before. What is new is what can be measured
around it:

1. **Is every premise necessary?** Drop each of the nine in turn.
2. **What did the English leave out?** Carroll's `old` and `young` are opposites to a
   reader and unrelated symbols to a solver. The file carries an extra constraint
   saying so. Remove it and watch what walks through the gap.
3. **What does variable ordering cost here?** Concept 3 made this point on a
   comparator built to make it. This formula was not built to make any point.

## 2. Definitions

### The puzzle

In [ ]:
# --- the second puzzle ---------------------------------------------------
# BDD/python/PyBool/examples/example_std_files/wise_young_pigs.txt.
# Nine premises, thirteen variables, one conclusion:
#     wise young pigs do not go up in balloons.
ORDER = '''
Var_Order : eatPennyBuns old young danceTightRopes
Var_Order : pigs respect giddy publicLunch ridiculous
Var_Order : umbrella fat wise balloon
'''

PREMISES = [
    'P1 = (~danceTightRopes & ~eatPennyBuns) => old',
    'P2 = (pigs & giddy) => respect',
    'P3 = (ridiculous & eatPennyBuns) => ~publicLunch',
    'P4 = (young & balloon) => giddy',
    'P5 = (wise & balloon) => umbrella',
    'P6 = (fat & ridiculous & ~danceTightRopes) => publicLunch',
    'P7 = (wise & giddy) => ~danceTightRopes',
    'P8 = (pigs & umbrella) => ridiculous',
    'P9 = (~danceTightRopes & respect) => fat',
]

# 'old' and 'young' are opposites to a reader and unrelated symbols to a
# solver.  Carroll's English never says so; the encoding has to.
EXTRA = 'Extra = old <=> ~young'
CONCL = 'C = (wise & young & pigs) => ~balloon'

def pig_spec(premises=PREMISES, extra=True, order=ORDER, refute=True):
    names = [p.split(' =')[0] for p in premises]
    lines = [order.strip()] + list(premises) + [EXTRA, CONCL]
    lines.append('P_All = ' + ' & '.join(names))
    main = 'P_All' + (' & Extra' if extra else '') + (' & ~C' if refute else '')
    lines.append('Main_Exp : ' + main)
    return '\n'.join(lines) + '\n'

<!-- nav-strip -->

---

&larr;&nbsp;[Ch17&nbsp;8.&nbsp;Lewis Carroll's Babies and Crocodiles, Decided by a Diagram](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter17-BDD/Concept-Carroll-Babies-And-Crocodiles/Concept-Carroll-Babies-And-Crocodiles.ipynb) &nbsp;&middot;&nbsp; [**Chapter 17** index](https://github.com/ganeshutah/Jove/blob/master/Chapter17-BDD/README.md) &nbsp;&middot;&nbsp; [Ch18&nbsp;1.&nbsp;The History of Lambda Calculus, and its Independence from Turing's Work](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter18-Lambda/Concept-History-Of-Lambda/Concept-History-Of-Lambda.ipynb)&nbsp;&rarr;

---

## 3. Tests

The premises first: consistent, and far too many models to eyeball.

In [ ]:
prem = bdd(pig_spec(refute=False))
print('variables              :', len(prem.vars))
print('assignments in total   :', 2 ** len(prem.vars))
print('models of the premises :', prem.count)
print('nodes                  :', prem.nodes)
assert prem.count > 0

**The refutation.**

In [ ]:
ref = bdd(pig_spec())
print('satisfying assignments :', ref.count)
print('nodes in the diagram   :', ref.nodes)
assert ref.count == 0
print()
print('So the nine premises entail it: wise young pigs do not go up in balloons.')

Thirteen variables in, one node out.

In [ ]:
ref

**Is every premise necessary?** Nine refutations, one premise missing from each.

In [ ]:
for i in range(len(PREMISES)):
    kept = PREMISES[:i] + PREMISES[i + 1:]
    b = bdd(pig_spec(kept))
    print('  without P%d : %s' % (i + 1,
          'still proved'
          if b.count == 0 else
          'NOT proved -- %d countermodel%s' % (b.count, '' if b.count == 1 else 's')))

Not one of them is redundant.

In [ ]:
print("Carroll left no slack in it.  That is a claim about a Victorian")
print("parlour puzzle, and nine BDDs just checked it.")
print()
print("The counts differ -- some premises rule out one stray world, others")
print("thirteen -- which is a rough measure of how much work each is doing.")

**What the English did not say.** `old` and `young` are opposites to you. To the solver they are two unrelated symbols, so the puzzle file adds `Extra = old <=> ~young`. Take it away.

In [ ]:
noextra = bdd(pig_spec(extra=False))
print('without the old/young axiom :', noextra.count, 'countermodel(s)')
m = noextra.models[0]
true_here = sorted(k for k, v in m.items() if v == 1)
print()
print('the countermodel sets these true:')
print('   ', ', '.join(true_here))
print()
print('   old   =', m['old'], '  young =', m['young'])

A pig that is both old and young.

In [ ]:
print("Nothing in the nine premises forbids it, because nothing in them")
print("says what 'old' and 'young' mean.  Carroll's reader supplies that")
print("for free and never notices doing it.")
print()
print("This is the gap between a sentence and its formalisation, and it is")
print("where real verification effort goes: not proving the theorem, but")
print("noticing the assumption you forgot to write down.")

**What does ordering cost?** The same formula, three orders.

In [ ]:
flat = [v for line in ORDER.strip().splitlines()
        for v in line.split(':')[1].split()]

def prem_nodes(order_vars):
    return bdd(pig_spec(order='Var_Order : ' + ' '.join(order_vars) + '\n',
                        refute=False)).nodes

print('%-26s %s' % ("the puzzle file's order", prem_nodes(flat)))
print('%-26s %s' % ('reversed', prem_nodes(list(reversed(flat)))))
print('%-26s %s' % ('alphabetical', prem_nodes(sorted(flat))))
print()
for label, order in (('file order', flat), ('alphabetical', sorted(flat))):
    b = bdd(pig_spec(order='Var_Order : ' + ' '.join(order) + '\n'))
    print('refutation, %-14s %d node(s)' % (label + ':', b.nodes))

The last two lines are the point.

In [ ]:
print("The premise diagram ranges from 97 nodes to 149 depending on an")
print("arbitrary-looking choice.  The REFUTATION is one node in every")
print("order, because false is false and canonicity leaves it nowhere to")
print("hide.")
print()
print("Ordering never changes the answer.  It changes what it costs to")
print("reach it -- and at Concept 7's scale, that is the whole game.")

## 4. Exercises


1. Which premise opened thirteen countermodels when removed? Read it, and say in a
   sentence why it constrains so much more than the others.
2. Add `young <=> ~old` as a *second* copy of the extra axiom. Does the node count
   change? Should it?
3. Invent a tenth premise that is *implied* by the other nine, add it, and confirm the
   refutation is unchanged. Then invent one that is not implied and see what moves.
4. Try ten random variable orders and report the best and worst premise node counts.
   How far is the puzzle file's order from the best you found?
5. Encode the crocodile puzzle from the previous notebook in this file's style, with a
   `P_All`. Confirm you get the same answer.
6. The refutation is one node whatever the ordering, but it still took work to build.
   Time it under the best and worst orders you found in exercise 4.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter17-BDD/Concept-Carroll-Wise-Young-Pigs')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')